# Validation Against Official Statistics — World Bank / IEA

**Author:** Bouchra Daddaoui  
**Repository:** viirs-electrification-ml  

This notebook validates the VIIRS NTL-based electricity access estimates from Notebook 11
against official national statistics from the World Bank World Development Indicators (WDI)
and the International Energy Agency (IEA) Africa Energy Outlook.

Replicating the validation approach of Echchabi et al. (2024) for SDG 6, we produce:
- Scatter plot: NTL-modeled % vs official % (with R², slope, 1:1 line)
- Urban/rural disaggregation comparison
- Residual analysis (systematic bias over time)
- Explicit limitation discussion

**Official data sources:**
- World Bank WDI indicator `EG.ELC.ACCS.ZS` (access to electricity, % of population)
- IEA Africa Energy Outlook 2022 (Morocco sub-national)
- ONEE Morocco annual reports (rural electrification)

**Outputs:**
- `figures/validation_scatter.png` — main validation figure
- `figures/validation_residuals.png` — temporal residual analysis
- `figures/validation_urban_rural.png` — urban/rural comparison

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import linregress, pearsonr
from shapely.geometry import box
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sdg7_tracker import (
    access_rate_timeseries, urban_rural_disaggregation,
    NTL_ELECTRIFICATION_THRESHOLD, COUNTRY_COLORS,
)
from temporal_analysis import mann_kendall

FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)
YEARS = list(range(2015, 2024))
print('Validation notebook loaded.')

## 1. Official Statistics

Hardcoded from IEA 2023 World Energy Outlook and World Bank WDI (EG.ELC.ACCS.ZS).
All values are national-level population percentages.

In [ ]:
# Official electricity access rates (% of population) — World Bank WDI / IEA
OFFICIAL_STATS = {
    'Brazil': {
        'national': {2015: 0.990, 2016: 0.991, 2017: 0.993, 2018: 0.995,
                     2019: 0.996, 2020: 0.997, 2021: 0.997, 2022: 0.997, 2023: 0.997},
        'urban':    {2020: 0.999, 2023: 0.999},
        'rural':    {2020: 0.975, 2023: 0.976},
        'source':   'World Bank WDI EG.ELC.ACCS.ZS',
    },
    'China': {
        'national': {yr: 1.000 for yr in range(2015, 2024)},
        'urban':    {2020: 1.000, 2023: 1.000},
        'rural':    {2020: 1.000, 2023: 1.000},
        'source':   'World Bank WDI / IEA 2023',
    },
    'Morocco': {
        'national': {2015: 0.968, 2016: 0.972, 2017: 0.975, 2018: 0.977,
                     2019: 0.979, 2020: 0.981, 2021: 0.983, 2022: 0.984, 2023: 0.985},
        'urban':    {2020: 0.998, 2023: 0.999},
        'rural':    {2020: 0.930, 2023: 0.950},
        'source':   'IEA Africa Energy Outlook 2022 + ONEE Morocco',
    },
}

print('Official Statistics Summary (National Level):')
for country, data in OFFICIAL_STATS.items():
    vals = list(data['national'].values())
    print(f"  {country:10s}: {min(vals)*100:.1f}% (2015) → {max(vals)*100:.1f}% (2023) | {data['source']}")

## 2. NTL-Modeled Access Rates (Notebook 11 Pipeline)

Rebuild the same synthetic panel and access rates for direct comparison.

In [ ]:
def generate_temporal_panel(country, n_tiles, bbox, ntl_base, ntl_std,
                             mean_growth, growth_std, shocks, seed, years):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = bbox
    side = int(np.sqrt(n_tiles))
    xs = np.linspace(minx, maxx, side + 1)
    ys = np.linspace(miny, maxy, side + 1)
    geoms, tile_ids = [], []
    for i in range(side):
        for j in range(side):
            geoms.append(box(xs[j], ys[i], xs[j+1], ys[i+1]))
            tile_ids.append(f'{country[:3].upper()}_{i:02d}_{j:02d}')
    n = len(geoms)
    centroids = np.array([[g.centroid.x, g.centroid.y] for g in geoms])
    dists = np.linalg.norm(centroids[:, None] - centroids[None, :], axis=-1)
    cov = ntl_std**2 * np.exp(-dists / (0.3 * (maxx - minx)))
    ntl_base_tile = np.clip(rng.multivariate_normal(np.full(n, ntl_base), cov), 0.5, None)
    growth_cov = growth_std**2 * np.exp(-dists / (0.5 * (maxx - minx)))
    tile_growth = rng.multivariate_normal(np.full(n, mean_growth), growth_cov)
    pop = rng.lognormal(mean=np.log(100), sigma=1.2, size=n)
    rows, ntl_curr = [], ntl_base_tile.copy()
    for yr in years:
        shock = shocks.get(yr, 0)
        ntl_curr = np.clip(ntl_curr + tile_growth + rng.normal(0, 0.3, n) + shock, 0, None)
        for i, tid in enumerate(tile_ids):
            rows.append({'tile_id': tid, 'country': country, 'year': yr,
                         'ntl_mean': ntl_curr[i], 'pop_density': pop[i],
                         'lon': centroids[i, 0], 'lat': centroids[i, 1]})
    return pd.DataFrame(rows), gpd.GeoDataFrame(
        {'tile_id': tile_ids, 'country': country, 'geometry': geoms}, crs='EPSG:4326')


country_params = [
    dict(country='Brazil',  n_tiles=100, bbox=(-48,-23,-43,-18), ntl_base=9.5,  ntl_std=6.0,
         mean_growth=0.42, growth_std=0.15, shocks={}, seed=1, years=YEARS),
    dict(country='China',   n_tiles=100, bbox=(116,29,122,33),   ntl_base=21.0, ntl_std=12.0,
         mean_growth=1.85, growth_std=0.40, shocks={2020: -3.1}, seed=2, years=YEARS),
    dict(country='Morocco', n_tiles=100, bbox=(-17.1,20.8,-1.0,35.9), ntl_base=6.2, ntl_std=4.0,
         mean_growth=0.38, growth_std=0.12, shocks={2017: 1.5}, seed=3, years=YEARS),
]

panel_dfs, geom_gdfs = [], {}
for params in country_params:
    df, gdf = generate_temporal_panel(**params)
    panel_dfs.append(df)
    geom_gdfs[params['country']] = gdf
panel_df = pd.concat(panel_dfs, ignore_index=True)

access_df = access_rate_timeseries(panel_df, threshold=NTL_ELECTRIFICATION_THRESHOLD)

print('NTL-modeled access rates (2015-2023):')
print(access_df.pivot(index='country', columns='year', values='access_rate').mul(100).round(1).to_string())

## 3. Validation Scatter Plot

X-axis: NTL-modeled access rate (%). Y-axis: Official (WB/IEA) access rate (%).  
27 points (3 countries × 9 years). R² and slope annotated.

In [ ]:
# Assemble matched pairs
pairs = []
for country, data in OFFICIAL_STATS.items():
    for yr, official_rate in data['national'].items():
        modeled = access_df[
            (access_df.country == country) & (access_df.year == yr)
        ]['access_rate'].values
        if len(modeled) > 0:
            pairs.append({
                'country': country, 'year': yr,
                'modeled': modeled[0], 'official': official_rate,
            })

pairs_df = pd.DataFrame(pairs)

# OLS regression
x = pairs_df['modeled'].values
y = pairs_df['official'].values
slope, intercept, r, p, se = linregress(x, y)
r2 = r**2

print(f'Regression: official = {slope:.3f} × modeled + {intercept:.3f}')
print(f'R² = {r2:.3f} | p = {p:.4f} | n = {len(pairs_df)}')

# Figure
fig, ax = plt.subplots(figsize=(7, 6))
for country in ['Brazil', 'China', 'Morocco']:
    sub = pairs_df[pairs_df.country == country]
    ax.scatter(sub.modeled * 100, sub.official * 100,
               color=COUNTRY_COLORS[country], s=60, label=country,
               edgecolors='white', linewidths=0.5, zorder=4)
    # Annotate year for first and last
    for _, row in sub[sub.year.isin([2015, 2023])].iterrows():
        ax.annotate(str(row.year), (row.modeled*100, row.official*100),
                    fontsize=6.5, ha='left', va='bottom', color=COUNTRY_COLORS[country])

# Regression line
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line * 100, (slope * x_line + intercept) * 100,
        '-', color='#333', linewidth=1.8, label=f'OLS fit (R²={r2:.2f}, slope={slope:.2f})')

# 1:1 identity line
lims = [min(x.min(), y.min()) * 100, max(x.max(), y.max()) * 100]
ax.plot(lims, lims, '--', color='grey', linewidth=1.2, alpha=0.7, label='1:1 line')

ax.set_xlabel('NTL-Modeled Access Rate (%)', fontsize=11)
ax.set_ylabel('Official Access Rate — WB/IEA (%)', fontsize=11)
ax.set_title('Validation: NTL-Based vs Official Electricity Access Estimates\n'
             f'n={len(pairs_df)} country-year pairs | R²={r2:.3f} | slope={slope:.3f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / 'validation_scatter.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: figures/validation_scatter.png')

## 4. Urban / Rural Disaggregation Comparison

In [ ]:
snapshot_2023 = panel_df[panel_df.year == 2023].copy()

ur_rows = []
for country in ['Brazil', 'China', 'Morocco']:
    grp = snapshot_2023[snapshot_2023.country == country]
    gdf_snap = gpd.GeoDataFrame(grp.reset_index(drop=True),
                                 geometry=geom_gdfs[country].geometry.values, crs='EPSG:4326')
    ur = urban_rural_disaggregation(gdf_snap)
    ur.insert(0, 'Country', country)
    ur['Official (%)'] = ur.apply(lambda r: (
        OFFICIAL_STATS[country].get('urban', {}).get(2023, np.nan) * 100
        if r['Stratum'] == 'Urban'
        else OFFICIAL_STATS[country].get('rural', {}).get(2023, np.nan) * 100
    ), axis=1)
    ur['Bias (pp)'] = (ur['Access Rate (%)'] - ur['Official (%)']).round(1)
    ur_rows.append(ur)

ur_df = pd.concat(ur_rows, ignore_index=True)
print('Urban/Rural Disaggregation Validation (2023):')
print(ur_df.to_string(index=False))

# Visualise
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(ur_df))
labels = [f"{r['Country']}\n{r['Stratum']}" for _, r in ur_df.iterrows()]
bars_m = ax.bar(x_pos - 0.2, ur_df['Access Rate (%)'], 0.35,
                color='#2c7bb6', label='NTL Modeled', alpha=0.85)
bars_o = ax.bar(x_pos + 0.2, ur_df['Official (%)'], 0.35,
                color='#d7191c', label='Official (WB/IEA)', alpha=0.85)
ax.set_xticks(x_pos); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Access Rate (%)', fontsize=10)
ax.set_title('NTL-Modeled vs Official Urban/Rural Access Rates (2023)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.25, axis='y')
fig.tight_layout()
fig.savefig(FIGURES / 'validation_urban_rural.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: figures/validation_urban_rural.png')

## 5. Residual Analysis — Systematic Bias Over Time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, country in zip(axes, ['Brazil', 'China', 'Morocco']):
    col = COUNTRY_COLORS[country]
    sub = pairs_df[pairs_df.country == country].sort_values('year')
    residuals = (sub.modeled - sub.official).values * 100  # pp
    yrs = sub.year.values

    ax.bar(yrs, residuals, color=[col if r >= 0 else '#d62728' for r in residuals],
           alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=1)

    # MK test on residuals
    mk = mann_kendall(residuals)
    trend_text = f"MK: {mk['trend']} (p={mk['p_value']:.3f})"
    ax.set_title(f'{country}\n{trend_text}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel('Residual (pp)')
    ax.set_xticks(yrs); ax.set_xticklabels(yrs, rotation=45, ha='right', fontsize=8)
    ax.grid(True, alpha=0.2)

    mean_bias = residuals.mean()
    ax.text(0.05, 0.95, f'Mean bias: {mean_bias:+.1f} pp',
            transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Residuals: NTL Model − Official Statistics (percentage points)\n'
             'Positive = model overestimates; Negative = underestimates',
             fontsize=11, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES / 'validation_residuals.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: figures/validation_residuals.png')

## 6. Limitations and Methodological Caveats

### 6.1 NTL Threshold Sensitivity
The `fixed=2.0 nW/cm²/sr` threshold is a single operating point. The threshold sensitivity
analysis in Notebook 11 shows ±5–15pp variation across methods. This introduces uncertainty
that is larger than the 95% CI of the Theil-Sen projection for Morocco.

### 6.2 Ceiling Effect
China is effectively at 100% access in all years. This compresses the scatter plot's upper
range and artificially inflates R². The R² reported here is partially driven by China's
perfect agreement, not by the model's discriminating power in partially-electrified regions.

### 6.3 Western Sahara Coverage
Morocco's bbox extends to 20.8°N to include Western Sahara — a large, predominantly dark
territory (~550,000 km²) with a sparse population (~600,000). The many unlit tiles depress
NTL-modeled access rates relative to the official national figure which is dominated by
northern Morocco (~37M population). Population weighting partially corrects this, but
does not fully resolve the geographic mismatch.

### 6.4 Urban Proxy Circularity
When GHSL built-up data is unavailable, we use NTL ≥ 10 nW/cm²/sr as an urban proxy.
This is circular since NTL is also the electrification signal. Future work should incorporate
GHSL 100m built-up surface area from GEE for non-circular stratification.

### 6.5 Annual Compositing
VIIRS monthly composites averaged to annual may smooth intra-year events (festivals, COVID
lockdowns reducing industrial NTL) that are detectable at monthly resolution.